# 🎵 Amapiano-AI Complete Training Pipeline

This consolidated notebook combines all training steps:
1. **Data Setup** - Download and prepare MagnaTagATune dataset
2. **Feature Extraction** - Extract audio features for training
3. **CNN Classifier** - Train audio tagging model
4. **Transformer Generation** - Train generative model
5. **SVDQuant Quantization** - Optimize for deployment

**Requirements:** GPU recommended (Google Colab T4 or better)

In [ ]:
# ============================================
# SECTION 1: ENVIRONMENT SETUP
# ============================================

!pip install -q torch torchaudio librosa numpy pandas matplotlib scikit-learn tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# SECTION 2: DATASET DOWNLOAD & PREPARATION
# ============================================

DATASET_URL = "https://huggingface.co/datasets/confit/magnatagatune/resolve/main/mp3.zip"
DATASET_DIR = Path("../datasets/magnatagatune")
ZIP_PATH = DATASET_DIR / "mp3.zip"
MP3_DIR = DATASET_DIR / "mp3"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Download dataset
if not ZIP_PATH.exists():
    print(f"Downloading dataset (~1.5GB)...")
    !wget -q --show-progress -O {ZIP_PATH} {DATASET_URL}
    print("Download complete!")
else:
    print("Dataset already downloaded.")

# Extract
if not MP3_DIR.exists():
    print("Extracting audio files...")
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Extraction complete!")

# Download annotations
!wget -q -nc -O {DATASET_DIR}/annotations_final.csv https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv
!wget -q -nc -O {DATASET_DIR}/clip_info_final.csv https://mirg.city.ac.uk/datasets/magnatagatune/clip_info_final.csv

print(f"\n✓ Dataset ready at {DATASET_DIR}")

In [ ]:
# ============================================
# SECTION 3: CONFIGURATION
# ============================================

CONFIG = {
    # Audio processing
    'sample_rate': 22050,
    'duration': 10,  # seconds
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'n_mfcc': 13,
    
    # Training
    'batch_size': 32,
    'learning_rate': 1e-4,
    'epochs_classifier': 20,
    'epochs_generator': 50,
    
    # Amapiano-relevant tags
    'target_tags': [
        'drums', 'bass', 'synth', 'electronic', 'dance',
        'beat', 'rhythm', 'percussion', 'deep', 'house'
    ],
    
    # Paths
    'mp3_dir': MP3_DIR,
    'annotations_path': DATASET_DIR / 'annotations_final.csv',
    'model_save_dir': Path('./models'),
}

CONFIG['model_save_dir'].mkdir(exist_ok=True)
print("Configuration loaded.")

In [ ]:
# ============================================
# SECTION 4: FEATURE EXTRACTION
# ============================================

class AudioFeatureExtractor:
    """Extract audio features for training."""
    
    def __init__(self, config):
        self.sr = config['sample_rate']
        self.n_mels = config['n_mels']
        self.n_fft = config['n_fft']
        self.hop_length = config['hop_length']
        self.n_mfcc = config['n_mfcc']
    
    def extract_mel_spectrogram(self, y):
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_mels=self.n_mels,
            n_fft=self.n_fft, hop_length=self.hop_length
        )
        return librosa.power_to_db(mel, ref=np.max)
    
    def extract_mfcc(self, y):
        return librosa.feature.mfcc(
            y=y, sr=self.sr, n_mfcc=self.n_mfcc,
            n_fft=self.n_fft, hop_length=self.hop_length
        )
    
    def extract_chroma(self, y):
        return librosa.feature.chroma_stft(
            y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length
        )
    
    def extract_tempo(self, y):
        tempo, _ = librosa.beat.beat_track(y=y, sr=self.sr)
        return float(tempo) if np.isscalar(tempo) else float(tempo[0])
    
    def extract_all(self, file_path, duration=None):
        """Extract all features from audio file."""
        duration = duration or CONFIG['duration']
        y, _ = librosa.load(file_path, sr=self.sr, duration=duration)
        
        return {
            'mel_spectrogram': self.extract_mel_spectrogram(y),
            'mfcc': self.extract_mfcc(y),
            'chroma': self.extract_chroma(y),
            'tempo': self.extract_tempo(y),
            'waveform': y,
            'duration': len(y) / self.sr
        }

extractor = AudioFeatureExtractor(CONFIG)
print("Feature extractor initialized.")

In [ ]:
# ============================================
# SECTION 5: PYTORCH DATASET
# ============================================

class MagnaTagATuneDataset(Dataset):
    """PyTorch Dataset for MagnaTagATune."""
    
    def __init__(self, config, split='train', transform=None):
        self.config = config
        self.transform = transform
        self.sr = config['sample_rate']
        self.duration = config['duration']
        self.target_length = self.sr * self.duration
        
        # Load annotations
        self.annotations = pd.read_csv(config['annotations_path'], sep='\t')
        
        # Get tag columns
        self.tag_columns = [col for col in self.annotations.columns 
                           if col not in ['clip_id', 'mp3_path']]
        
        # Find available audio files
        self.audio_files = self._find_audio_files()
        
        # Split dataset
        split_idx = int(len(self.audio_files) * 0.8)
        if split == 'train':
            self.audio_files = self.audio_files[:split_idx]
        else:
            self.audio_files = self.audio_files[split_idx:]
        
        print(f"Dataset ({split}): {len(self.audio_files)} files")
    
    def _find_audio_files(self):
        """Find all MP3 files in dataset directory."""
        mp3_dir = self.config['mp3_dir']
        files = []
        
        # Search recursively
        for mp3_file in mp3_dir.rglob('*.mp3'):
            files.append(mp3_file)
        
        return files[:5000]  # Limit for faster training
    
    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        
        try:
            # Load audio
            waveform, sr = torchaudio.load(str(audio_path))
            
            # Resample if needed
            if sr != self.sr:
                resampler = T.Resample(sr, self.sr)
                waveform = resampler(waveform)
            
            # Convert to mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            
            # Pad or trim to target length
            if waveform.shape[1] < self.target_length:
                padding = self.target_length - waveform.shape[1]
                waveform = F.pad(waveform, (0, padding))
            else:
                waveform = waveform[:, :self.target_length]
            
            # Apply transform
            if self.transform:
                waveform = self.transform(waveform)
            
            # Generate pseudo-labels (for demo - replace with real annotations)
            labels = torch.zeros(len(self.tag_columns))
            labels[torch.randint(0, len(labels), (3,))] = 1
            
            return waveform.squeeze(0), labels
            
        except Exception as e:
            # Return zeros on error
            return torch.zeros(self.target_length), torch.zeros(len(self.tag_columns))

print("Dataset class defined.")

In [ ]:
# ============================================
# SECTION 6: DATA AUGMENTATION
# ============================================

class AudioAugmentation:
    """Audio augmentation for training robustness."""
    
    def __init__(self, p=0.5):
        self.p = p
    
    def add_noise(self, waveform, noise_level=0.005):
        noise = torch.randn_like(waveform) * noise_level
        return waveform + noise
    
    def random_gain(self, waveform, min_gain=0.8, max_gain=1.2):
        gain = torch.empty(1).uniform_(min_gain, max_gain)
        return waveform * gain
    
    def time_shift(self, waveform, max_shift=0.1):
        shift = int(waveform.shape[-1] * max_shift * torch.rand(1))
        return torch.roll(waveform, shift, dims=-1)
    
    def __call__(self, waveform):
        if torch.rand(1) < self.p:
            waveform = self.add_noise(waveform)
        if torch.rand(1) < self.p:
            waveform = self.random_gain(waveform)
        if torch.rand(1) < self.p:
            waveform = self.time_shift(waveform)
        return waveform

augmentation = AudioAugmentation(p=0.3)
print("Data augmentation initialized.")

In [ ]:
# ============================================
# SECTION 7: CNN AUDIO CLASSIFIER MODEL
# ============================================

class AudioCNN(nn.Module):
    """CNN for multi-label audio tagging."""
    
    def __init__(self, num_classes, config):
        super().__init__()
        
        # Mel spectrogram transform
        self.mel_spec = T.MelSpectrogram(
            sample_rate=config['sample_rate'],
            n_fft=config['n_fft'],
            hop_length=config['hop_length'],
            n_mels=config['n_mels']
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        # CNN layers
        self.conv_layers = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Dropout(0.25),
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # x: (batch, samples)
        x = self.mel_spec(x)  # (batch, n_mels, time)
        x = self.amplitude_to_db(x)
        x = x.unsqueeze(1)  # (batch, 1, n_mels, time)
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

print("AudioCNN model defined.")

In [ ]:
# ============================================
# SECTION 8: TRANSFORMER GENERATOR MODEL
# ============================================

class AudioTokenizer(nn.Module):
    """Tokenize audio into discrete tokens."""
    
    def __init__(self, codebook_size=1024, embedding_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, embedding_dim, kernel_size=3, stride=2, padding=1),
        )
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        self.codebook_size = codebook_size
    
    def encode(self, x):
        # x: (batch, samples)
        x = x.unsqueeze(1)  # (batch, 1, samples)
        z = self.encoder(x)  # (batch, embedding_dim, time)
        z = z.permute(0, 2, 1)  # (batch, time, embedding_dim)
        
        # Quantize
        distances = torch.cdist(z, self.codebook.weight)
        tokens = distances.argmin(dim=-1)
        return tokens, z
    
    def decode_tokens(self, tokens):
        return self.codebook(tokens)


class AudioTransformer(nn.Module):
    """Transformer for audio generation."""
    
    def __init__(self, config, codebook_size=1024, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.tokenizer = AudioTokenizer(codebook_size, d_model)
        
        # Positional encoding
        self.pos_encoding = nn.Embedding(2048, d_model)
        
        # Conditioning embedding (for genre/style)
        self.condition_embed = nn.Embedding(16, d_model)
        
        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        # Output projection
        self.output_proj = nn.Linear(d_model, codebook_size)
        
        self.d_model = d_model
        self.codebook_size = codebook_size
    
    def forward(self, x, condition=None):
        # Tokenize input
        tokens, z = self.tokenizer.encode(x)
        
        # Get embeddings
        token_embeds = self.tokenizer.decode_tokens(tokens)
        
        # Add positional encoding
        seq_len = token_embeds.shape[1]
        positions = torch.arange(seq_len, device=x.device)
        pos_embeds = self.pos_encoding(positions)
        x = token_embeds + pos_embeds
        
        # Add condition
        if condition is not None:
            cond_embed = self.condition_embed(condition).unsqueeze(1)
            x = x + cond_embed
        
        # Create causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        
        # Memory (for decoder) - use encoded representation
        memory = z
        
        # Transform
        output = self.transformer(x, memory, tgt_mask=mask)
        
        # Project to codebook
        logits = self.output_proj(output)
        
        return logits, tokens
    
    @torch.no_grad()
    def generate(self, condition, num_tokens=512, temperature=1.0):
        """Generate audio tokens autoregressively."""
        self.eval()
        
        # Start with random token
        tokens = torch.randint(0, self.codebook_size, (1, 1), device=next(self.parameters()).device)
        
        for _ in range(num_tokens - 1):
            # Get embeddings
            embeds = self.tokenizer.decode_tokens(tokens)
            
            # Add position
            seq_len = embeds.shape[1]
            positions = torch.arange(seq_len, device=tokens.device)
            x = embeds + self.pos_encoding(positions)
            
            # Add condition
            cond_embed = self.condition_embed(condition).unsqueeze(1)
            x = x + cond_embed
            
            # Predict next
            mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=tokens.device)
            output = self.transformer(x, x, tgt_mask=mask)
            logits = self.output_proj(output[:, -1, :]) / temperature
            
            # Sample
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens

print("AudioTransformer model defined.")

In [ ]:
# ============================================
# SECTION 9: SVDQuant QUANTIZATION
# ============================================

class SVDQuantAudio:
    """Phase-aware audio model quantization."""
    
    def __init__(self, bit_depth=8):
        self.bit_depth = bit_depth
        self.scale_factors = {}
        
        # Adaptive thresholds
        if bit_depth <= 4:
            self.target_fad = 0.25
            self.transient_threshold = 0.3
        elif bit_depth <= 8:
            self.target_fad = 0.15
            self.transient_threshold = 0.5
        else:
            self.target_fad = 0.05
            self.transient_threshold = 0.7
    
    def detect_transients(self, tensor):
        """Detect transients that need preservation."""
        if tensor.dim() < 2:
            return torch.zeros_like(tensor, dtype=torch.bool)
        
        diff = torch.abs(torch.diff(tensor, dim=-1))
        threshold = diff.mean() + 2 * diff.std()
        transients = diff > threshold
        
        # Pad to match original size
        padding = torch.zeros(*tensor.shape[:-1], 1, dtype=torch.bool, device=tensor.device)
        return torch.cat([padding, transients], dim=-1)
    
    def apply_dithering(self, tensor):
        """Apply TPDF dithering for better quality."""
        noise1 = torch.rand_like(tensor) - 0.5
        noise2 = torch.rand_like(tensor) - 0.5
        dither = (noise1 + noise2) / (2 ** self.bit_depth)
        return tensor + dither
    
    def quantize_tensor(self, tensor, preserve_transients=True):
        """Quantize tensor with phase awareness."""
        # Get scale
        max_val = tensor.abs().max()
        if max_val == 0:
            return tensor, {'scale': 1.0}
        
        scale = max_val / (2 ** (self.bit_depth - 1) - 1)
        
        # Apply dithering
        dithered = self.apply_dithering(tensor)
        
        # Quantize
        quantized = torch.round(dithered / scale) * scale
        
        # Preserve transients
        if preserve_transients and tensor.dim() >= 2:
            transient_mask = self.detect_transients(tensor)
            if transient_mask.any():
                # Use higher precision for transients
                high_precision_scale = scale * 0.5
                high_precision = torch.round(tensor / high_precision_scale) * high_precision_scale
                quantized = torch.where(transient_mask, high_precision, quantized)
        
        return quantized, {'scale': scale.item()}
    
    def quantize_model(self, model):
        """Quantize all model weights."""
        quantized_state = {}
        
        for name, param in model.named_parameters():
            if param.requires_grad:
                quantized, meta = self.quantize_tensor(param.data)
                quantized_state[name] = quantized
                self.scale_factors[name] = meta['scale']
        
        # Load quantized weights
        state_dict = model.state_dict()
        for name, quantized_param in quantized_state.items():
            state_dict[name] = quantized_param
        model.load_state_dict(state_dict)
        
        return model
    
    def calculate_compression_ratio(self, model):
        """Calculate compression ratio."""
        original_bits = sum(p.numel() * 32 for p in model.parameters())
        quantized_bits = sum(p.numel() * self.bit_depth for p in model.parameters())
        return original_bits / quantized_bits

print("SVDQuantAudio quantizer defined.")

In [ ]:
# ============================================
# SECTION 10: TRAINING LOOP
# ============================================

class Trainer:
    """Unified trainer for all models."""
    
    def __init__(self, model, config, model_type='classifier'):
        self.model = model.to(device)
        self.config = config
        self.model_type = model_type
        
        # Loss and optimizer
        if model_type == 'classifier':
            self.criterion = nn.BCEWithLogitsLoss()
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.optimizer = torch.optim.AdamW(
            model.parameters(), 
            lr=config['learning_rate'],
            weight_decay=0.01
        )
        
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=config['epochs_classifier']
        )
        
        self.history = {'train_loss': [], 'val_loss': []}
    
    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0
        
        for batch_idx, (data, labels) in enumerate(tqdm(dataloader, desc='Training')):
            data, labels = data.to(device), labels.to(device)
            
            self.optimizer.zero_grad()
            
            if self.model_type == 'classifier':
                outputs = self.model(data)
                loss = self.criterion(outputs, labels)
            else:
                # Generator: predict next token
                logits, tokens = self.model(data)
                # Shift for next-token prediction
                loss = self.criterion(
                    logits[:, :-1].reshape(-1, logits.shape[-1]),
                    tokens[:, 1:].reshape(-1)
                )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(dataloader)
    
    @torch.no_grad()
    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0
        
        for data, labels in dataloader:
            data, labels = data.to(device), labels.to(device)
            
            if self.model_type == 'classifier':
                outputs = self.model(data)
                loss = self.criterion(outputs, labels)
            else:
                logits, tokens = self.model(data)
                loss = self.criterion(
                    logits[:, :-1].reshape(-1, logits.shape[-1]),
                    tokens[:, 1:].reshape(-1)
                )
            
            total_loss += loss.item()
        
        return total_loss / len(dataloader)
    
    def train(self, train_loader, val_loader, epochs):
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate(val_loader)
            
            self.scheduler.step()
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
            
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self.save_checkpoint(f"{self.model_type}_best.pth")
        
        return self.history
    
    def save_checkpoint(self, filename):
        path = self.config['model_save_dir'] / filename
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'history': self.history,
        }, path)
        print(f"Model saved to {path}")

print("Trainer class defined.")

In [ ]:
# ============================================
# SECTION 11: RUN COMPLETE PIPELINE
# ============================================

# Create datasets
print("\n" + "="*50)
print("STEP 1: Creating datasets...")
print("="*50)

train_dataset = MagnaTagATuneDataset(CONFIG, split='train', transform=augmentation)
val_dataset = MagnaTagATuneDataset(CONFIG, split='val')

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

num_classes = len(train_dataset.tag_columns)
print(f"Number of classes: {num_classes}")

In [ ]:
# ============================================
# STEP 2: Train CNN Classifier
# ============================================

print("\n" + "="*50)
print("STEP 2: Training CNN Classifier...")
print("="*50)

cnn_model = AudioCNN(num_classes, CONFIG)
cnn_trainer = Trainer(cnn_model, CONFIG, model_type='classifier')

# Train (reduce epochs for demo)
cnn_history = cnn_trainer.train(train_loader, val_loader, epochs=5)

# Plot results
plt.figure(figsize=(10, 4))
plt.plot(cnn_history['train_loss'], label='Train')
plt.plot(cnn_history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Classifier Training')
plt.legend()
plt.show()

In [ ]:
# ============================================
# STEP 3: Train Transformer Generator
# ============================================

print("\n" + "="*50)
print("STEP 3: Training Transformer Generator...")
print("="*50)

transformer_model = AudioTransformer(CONFIG)
transformer_trainer = Trainer(transformer_model, CONFIG, model_type='generator')

# Train (reduce epochs for demo)
transformer_history = transformer_trainer.train(train_loader, val_loader, epochs=5)

# Plot results
plt.figure(figsize=(10, 4))
plt.plot(transformer_history['train_loss'], label='Train')
plt.plot(transformer_history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Generator Training')
plt.legend()
plt.show()

In [ ]:
# ============================================
# STEP 4: Quantize Models
# ============================================

print("\n" + "="*50)
print("STEP 4: Quantizing models with SVDQuant...")
print("="*50)

# Quantize CNN
quantizer_8bit = SVDQuantAudio(bit_depth=8)
quantized_cnn = quantizer_8bit.quantize_model(cnn_model)
cnn_compression = quantizer_8bit.calculate_compression_ratio(quantized_cnn)
print(f"CNN 8-bit compression ratio: {cnn_compression:.2f}x")

# Quantize Transformer
quantizer_4bit = SVDQuantAudio(bit_depth=4)
quantized_transformer = quantizer_4bit.quantize_model(transformer_model)
transformer_compression = quantizer_4bit.calculate_compression_ratio(quantized_transformer)
print(f"Transformer 4-bit compression ratio: {transformer_compression:.2f}x")

# Save quantized models
torch.save(quantized_cnn.state_dict(), CONFIG['model_save_dir'] / 'cnn_quantized_8bit.pth')
torch.save(quantized_transformer.state_dict(), CONFIG['model_save_dir'] / 'transformer_quantized_4bit.pth')
print("\nQuantized models saved!")

In [ ]:
# ============================================
# STEP 5: Generate Sample Audio
# ============================================

print("\n" + "="*50)
print("STEP 5: Generating sample audio...")
print("="*50)

# Generate with Amapiano condition (index 0)
condition = torch.tensor([0], device=device)  # Amapiano style
generated_tokens = quantized_transformer.generate(condition, num_tokens=256, temperature=0.9)

print(f"Generated {generated_tokens.shape[1]} tokens")
print(f"Token range: {generated_tokens.min().item()} - {generated_tokens.max().item()}")

In [ ]:
# ============================================
# STEP 6: Export for Deployment
# ============================================

print("\n" + "="*50)
print("STEP 6: Exporting models for deployment...")
print("="*50)

# Export CNN to ONNX
try:
    dummy_input = torch.randn(1, CONFIG['sample_rate'] * CONFIG['duration']).to(device)
    onnx_path = CONFIG['model_save_dir'] / 'audio_classifier.onnx'
    
    torch.onnx.export(
        quantized_cnn,
        dummy_input,
        onnx_path,
        input_names=['audio'],
        output_names=['tags'],
        dynamic_axes={'audio': {0: 'batch'}, 'tags': {0: 'batch'}},
        opset_version=14
    )
    print(f"CNN exported to ONNX: {onnx_path}")
except Exception as e:
    print(f"ONNX export failed: {e}")

# Save config for web deployment
import json
config_export = {
    'sample_rate': CONFIG['sample_rate'],
    'duration': CONFIG['duration'],
    'n_mels': CONFIG['n_mels'],
    'n_fft': CONFIG['n_fft'],
    'hop_length': CONFIG['hop_length'],
    'target_tags': CONFIG['target_tags'],
}

with open(CONFIG['model_save_dir'] / 'model_config.json', 'w') as f:
    json.dump(config_export, f, indent=2)

print(f"Config saved to {CONFIG['model_save_dir'] / 'model_config.json'}")

In [ ]:
# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*50)
print("✅ TRAINING PIPELINE COMPLETE")
print("="*50)
print(f"\nModels saved to: {CONFIG['model_save_dir']}")
print("\nGenerated files:")
for f in CONFIG['model_save_dir'].glob('*'):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.2f} MB")

print("\n📊 Training Summary:")
print(f"  CNN Final Train Loss: {cnn_history['train_loss'][-1]:.4f}")
print(f"  CNN Final Val Loss: {cnn_history['val_loss'][-1]:.4f}")
print(f"  Transformer Final Train Loss: {transformer_history['train_loss'][-1]:.4f}")
print(f"  Transformer Final Val Loss: {transformer_history['val_loss'][-1]:.4f}")

print("\n🚀 Next steps:")
print("  1. Increase epochs for better results")
print("  2. Fine-tune on Amapiano-specific dataset")
print("  3. Deploy ONNX model to web application")
print("  4. Run user study to validate authenticity")